# 檢索排序與機率融合

[完整講義](https://github.com/Samuel-ncu/AI-Stastic/blob/main/docs/units/e3-retrieval-probability.md) · [在 Colab 執行](https://colab.research.google.com/github/Samuel-ncu/AI-Stastic/blob/main/notebooks/e3_retrieval_probability.ipynb)

本練習使用自編小型資料，在 CPU 執行。讀出兩個查詢的覆蓋率與第一份相關文件的名次，再核對機率融合。

RAG 先檢索文件，再根據文件產生回答。這兩步各有不同的問題：相關文件是否排得夠前？文件提供的證據如何影響答案？

## 從餘弦相似度到 Recall@k 與 MRR

沿用第 10 課的餘弦相似度。查詢 $q=[1,0]$，三份文件向量分別是 $D_1=[1,0]$、$D_2=[1,1]$、$D_3=[0,1]$。與查詢的餘弦相似度為 1、$1/\sqrt2$、0，排序為 $D_1,D_2,D_3$。假設人工標記的相關文件是 $D_2,D_3$。

| 指標 | 回答的問題 | 本例 |
|---|---|---|
| Recall@2 | 全部相關文件中，有多少出現在前兩名？ | 找到 $D_2$，為 $1/2=0.5$ |
| Reciprocal Rank（RR） | 第一份相關文件排第幾？ | $D_2$ 排第 2，為 $1/2=0.5$ |

MRR 是多個查詢的 RR 平均。假設第二個查詢三份文件都相關，前兩名找到兩份，Recall@2 為 $2/3$。第一份相關文件排第 1，所以第二題的 RR 為 1，兩個查詢的 MRR 就是 $(0.5+1)/2=0.75$。第二題也說明，第一名就命中時，仍可能有部分相關文件沒進入前兩名。本練習把無命中的 RR 記為 0，且每個查詢都有至少一份已標註的相關文件。

## 如何把文件權重帶進答案機率？

為了理解 RAG 的邊際化，假設兩份文件已有歸一化權重 0.6、0.4。它們各自讓某答案成立的條件機率為 0.2、0.9。用文件權重加權後，答案機率是

$$0.6\times0.2+0.4\times0.9=0.48.$$

這個運算沿用期望值的加權平均。上節餘弦分數用來排序；這節的權重已事先指定為總和 1 的機率。評估時再問「拿到相關文件後，答案有沒有答對」，因為文件的排名和最終答案各需要一套判準。

## 執行環境

Colab 通常已有 NumPy 和 Matplotlib。若要使用本專案規定範圍內的套件，可執行下一格。

In [ ]:
%pip -q install 'numpy>=1.26,<3' 'matplotlib>=3.8,<4'

## 程式驗證

先預測結果，再執行下方的手算核對和視覺化。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

documents = np.array([[1., 0.], [1., 1.], [0., 1.]])
queries = np.array([[1., 0.], [0., 1.]])
relevant = [{1, 2}, {0, 1, 2}]
document_norms = np.linalg.norm(documents, axis=1)
scores = (queries @ documents.T) / (np.linalg.norm(queries, axis=1)[:, None] * document_norms)
rankings = np.argsort(-scores, axis=1, kind="stable")

k = 2
recalls = [len(set(ranking[:k]) & labels) / len(labels) for ranking, labels in zip(rankings, relevant)]
rr = [
    next((1 / rank for rank, doc in enumerate(ranking, 1) if doc in labels), 0.0)
    for ranking, labels in zip(rankings, relevant)
]
merged_probability = np.dot([0.6, 0.4], [0.2, 0.9])
np.testing.assert_allclose(scores[0], [1, 1 / np.sqrt(2), 0])
np.testing.assert_allclose(recalls, [0.5, 2 / 3])
np.testing.assert_allclose(np.mean(rr), 0.75)
np.testing.assert_allclose(merged_probability, 0.48)
print("Rankings, starting at D1:", (rankings + 1).tolist())
print("Recall@2:", recalls, "MRR:", np.mean(rr))
print("Marginal answer probability:", merged_probability)
fig, ax = plt.subplots()
locations = np.arange(len(queries))
ax.bar(locations - 0.18, recalls, 0.35, label="Recall@2")
ax.bar(locations + 0.18, rr, 0.35, label="Reciprocal rank")
ax.set(xticks=locations, xticklabels=["Query 1", "Query 2"], ylim=(0, 1.15), title="Coverage versus first relevant rank")
ax.legend()
plt.show()

## 自我檢核與練習

把第一題的 $k$ 改為 3。預測 Recall@k 和 RR 各怎麼變。參考答案：Recall@3 變成 1，第一份相關文件仍在第 2 名，所以 RR 維持 0.5。

## 來源

- [RAG 原論文](https://arxiv.org/abs/2005.11401)
- [LLM-Embedder 原論文](https://arxiv.org/abs/2310.07554)